In [1]:
import numpy as np
import pandas as pd

In [2]:
print("Loading datasets...")
results = pd.read_csv('../data/raw/results.csv')
shootouts = pd.read_csv('../data/raw/shootouts.csv')
former_names = pd.read_csv('../data/raw/former_names.csv')
rankings = pd.read_csv('../data/raw/archive/fifa_ranking-2024-06-20.csv')

Loading datasets...


In [3]:
# Convert date columns to datetime format
results['date'] = pd.to_datetime(results['date'])
shootouts['date'] = pd.to_datetime(shootouts['date'])
rankings['rank_date'] = pd.to_datetime(rankings['rank_date'])

In [4]:
print("Harmonizing team names across datasets...")

# Create mapping dictionary from former_names
name_mapping = dict(zip(former_names['former'], former_names['current']))

# Supplement mapping for inconsistencies between results and historical rankings
name_mapping.update({
    "United States": "USA",
    "US Virgin Islands": "USA Virgin Islands",
    "St. Kitts and Nevis": "St Kitts and Nevis",
    "St. Lucia": "St Lucia",
    "St. Vincent and the Grenadines": "St Vincent / Grenadines",
    "Antigua and Barbuda": "Antigua & Barbuda",
    "Trinidad and Tobago": "Trinidad and Tobago",
    "South Korea": "Korea Republic",
    "North Korea": "Korea DPR",
    "China PR": "China",
    "Côte d'Ivoire": "Ivory Coast",
    "Republic of Ireland": "Eire",
    "Turkey": "Türkiye",
    "Congo DR": "DR Congo",
    "Zaïre": "DR Congo",
    "IR Iran": "Iran",
    "Iran IR": "Iran",
    "Cabo Verde": "Cape Verde",
    "Cape Verde Islands": "Cape Verde",
    "Democratic Republic of the Congo": "DR Congo",
    "Korea DPR": "North Korea",
    'Czechia': 'Czech Republic'
})


Harmonizing team names across datasets...


In [5]:
# Function to clean team names in a given dataframe and specified columns
def clean_names(df, columns, target_mapping):
    for col in columns:
        if col in df.columns:
            # 1. Use the .str accessor to safely strip any accidental leading/trailing spaces
            df[col] = df[col].astype(str).str.strip()
            # 2. Replace the names using your dictionary mapping
            df[col] = df[col].replace(target_mapping)
            
    return df

In [6]:
# Clean team names in all datasets
results = clean_names(results, ['home_team', 'away_team'], name_mapping)
rankings = clean_names(rankings, ['country_full'], name_mapping)

results = results.sort_values('date').reset_index(drop=True)
rankings = rankings.sort_values('rank_date').reset_index(drop=True)

In [7]:
print("Executing point-in-time merge for historical rankings...")

# Select only the columns we need from the ranking dataset to avoid bloat
rank_lookup = rankings[['rank_date', 'country_full', 'rank', 'total_points']].copy()

# Merge for the Home Team
results = pd.merge_asof(
    results,
    rank_lookup,
    left_on='date',
    right_on='rank_date',
    left_by='home_team',
    right_by='country_full',
    direction='backward' # Looks for the latest ranking on or before the match date
)
results = results.rename(columns={'rank': 'home_fifa_rank', 'total_points': 'home_fifa_points'})
results = results.drop(columns=['rank_date', 'country_full'])

# Merge for the Away Team
results = pd.merge_asof(
    results,
    rank_lookup,
    left_on='date',
    right_on='rank_date',
    left_by='away_team',
    right_by='country_full',
    direction='backward'
)
results = results.rename(columns={'rank': 'away_fifa_rank', 'total_points': 'away_fifa_points'})
results = results.drop(columns=['rank_date', 'country_full'])

Executing point-in-time merge for historical rankings...


In [8]:
print("Encoding match outcomes and structural features...")

# Create target variable: home_win (0), draw (1), away_win (2)
conditions = [
    (results['home_score'] > results['away_score']),
    (results['home_score'] == results['away_score']),
    (results['home_score'] < results['away_score'])
]
choices = [0, 1, 2] # Numerical classes work beautifully with XGBoost
results['outcome'] = np.select(conditions, choices, default=np.nan)

# Generate baseline context features
results['rank_diff'] = results['home_fifa_rank'] - results['away_fifa_rank']
results['point_diff'] = results['home_fifa_points'] - results['away_fifa_points']
results['is_neutral'] = results['neutral'].astype(int)

# Save intermediate matrix to your processed data folder
results.to_csv('../data/processed/cleaned_historical_dataset.csv', index=False)
print("Phase 2 (Part 1) successfully generated: '../data/processed/cleaned_historical_dataset.csv'")
print(results[['date', 'home_team', 'away_team', 'home_fifa_rank', 'away_fifa_rank', 'outcome']].tail(10))

Encoding match outcomes and structural features...
Phase 2 (Part 1) successfully generated: '../data/processed/cleaned_historical_dataset.csv'
            date    home_team     away_team  home_fifa_rank  away_fifa_rank  \
49401 2026-06-26      Uruguay         Spain            14.0             8.0   
49402 2026-06-26  New Zealand       Belgium           107.0             3.0   
49403 2026-06-26        Egypt          Iran            36.0            20.0   
49404 2026-06-26   Cape Verde  Saudi Arabia            65.0            56.0   
49405 2026-06-27       Panama       England            43.0             5.0   
49406 2026-06-27      Algeria       Austria            44.0            25.0   
49407 2026-06-27       Jordan     Argentina            68.0             1.0   
49408 2026-06-27     Colombia      Portugal            12.0             6.0   
49409 2026-06-27     DR Congo    Uzbekistan            61.0            62.0   
49410 2026-06-27      Croatia         Ghana             9.0        